In [1]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = duckdb.connect("../data/capstone_data.duckdb")

## For Nov and Dec

Spending aggregates

In [2]:
# To have an aggregate for transaction overview for each customer
spending_agg = con.execute("""
    SELECT 
        t.CustomerID,
        c.Individual,
        SUM(t.Amount_Completed)    AS total_spend,
        COUNT(*)                 AS transaction_count,
        AVG(t.Amount_Completed)    AS avg_transaction_amount,
        MEDIAN(t.Amount_Completed) AS median_transaction_amount,
        STDDEV(t.Amount_Completed) AS std_transaction_amount,
        MAX(t.Amount_Completed)    AS max_transaction_amount,
        MIN(t.Amount_Completed)    AS min_transaction_amount
    FROM transactions_v2 t
    INNER JOIN customers c on t.CustomerID = c.CustomerID
    WHERE t.Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25'
        AND t.Amount_Completed > 0
    GROUP BY t.CustomerID, c.Individual
""").fetchdf()

print(spending_agg.shape)
spending_agg.head()

(68585, 9)


,CustomerID,Individual,total_spend,transaction_count,avg_transaction_amount,median_transaction_amount,std_transaction_amount,max_transaction_amount,min_transaction_amount
0,BSB382953,Y,1915.99,70,27.371286,14.815,50.850889,389.97,1.04
1,BSB325340,Y,3189.02,91,35.044176,22.180,39.062109,200.00,0.73
2,BSB195798,Y,4484.05,174,25.770402,18.800,57.728915,750.00,0.99
3,BSB021620,Y,2259.61,48,47.075208,25.025,59.437927,263.00,3.30
4,BSB176785,Y,5809.03,125,46.472240,26.230,60.207840,273.42,0.50


In [3]:
spending_agg.describe()

,total_spend,transaction_count,avg_transaction_amount,median_transaction_amount,std_transaction_amount,max_transaction_amount,min_transaction_amount
count,68585.000000,68585.000000,68585.000000,68585.000000,65553.000000,68585.000000,68585.000000
mean,3179.545752,62.888970,70.933138,45.684600,88.193099,449.249026,16.870730
std,4323.445091,63.681175,190.227930,158.281723,246.447021,961.583718,126.561992
min,0.020000,1.000000,0.020000,0.020000,0.000000,0.020000,0.010000
25%,624.930000,12.000000,28.320000,16.030000,28.991878,120.000000,1.060000
50%,2061.260000,44.000000,43.968448,25.000000,51.688426,250.000000,2.680000
75%,4351.090000,95.000000,69.276957,40.000000,90.714963,492.050000,6.840000
max,225125.100000,1424.000000,19655.666667,11896.500000,37145.989928,95000.000000,11896.500000


Customers who made a transaction ->  68585 \
std null count --> 65553 (so 3000 customers made 1 transaction only)


Category shares

In [4]:
# PRINT ALL MCC CODES INTO A CSV
# all_mcc = con.execute("""
#     SELECT 
#         Merchant_Category,
#         COUNT(*) as txn_count
#     FROM transactions_v2
#     GROUP BY Merchant_Category
#     ORDER BY txn_count DESC
# """).fetchdf()

# all_mcc['mcc_label'] = ''  # empty column to fill in with actual value

# all_mcc.to_csv('../data/mcc_mapping.csv', index=False)
# print(f"Exported {len(all_mcc)} MCCs")

In [4]:
category_features = con.execute("""
    SELECT 
        t.CustomerID,
        c.Individual,
        -- each category share = that category's spend / total spend
        SUM(CASE WHEN m.Category = 'Grocery' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS grocery_share,
        SUM(CASE WHEN m.Category = 'Dining' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS dining_share,
        SUM(CASE WHEN m.Category = 'Gas' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS gas_share,
        SUM(CASE WHEN m.Category = 'Digital/Subsciptions' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS digital_share,
        SUM(CASE WHEN m.Category = 'Telecom/Cable' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS telecom_share,
        SUM(CASE WHEN m.Category = 'Retail' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS retail_share,
        SUM(CASE WHEN m.Category = 'Healthcare' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS healthcare_share,
        SUM(CASE WHEN m.Category = 'Pets' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS pets_share,
        SUM(CASE WHEN m.Category = 'Home/Hardware' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS home_share,
        SUM(CASE WHEN m.Category = 'Automotive' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS automotive_share,
        SUM(CASE WHEN m.Category = 'Clothing' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS clothing_share,
        SUM(CASE WHEN m.Category = 'Entertainment/Leisure' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS entertainment_share,
        SUM(CASE WHEN m.Category = 'Financial Services' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS financial_share,
        SUM(CASE WHEN m.Category = 'Personal Care' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS personal_care_share,
        SUM(CASE WHEN m.Category = 'Transportation' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS transport_share,
        SUM(CASE WHEN m.Category = 'Utilities' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS utilities_share,
        SUM(CASE WHEN m.Category = 'Government/Community' 
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS government_share,
        SUM(CASE WHEN m.Category = 'other' OR m.Category IS NULL
            THEN t.Amount_Completed ELSE 0 END) / SUM(t.Amount_Completed) AS other_share

    FROM transactions_v2 t
    INNER JOIN customers c ON t.CustomerID = c.CustomerID
    LEFT JOIN mcc_mapping m 
        ON CAST(t.Merchant_Category AS VARCHAR) = m.Merchant_Category
    WHERE t.Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25'
      AND t.Amount_Completed > 0
    GROUP BY t.CustomerID, c.Individual
""").fetchdf()


print(category_features.head(1).T)
print(f"Shape: {category_features.shape}")
print("\nShare totals per customer (should all be ~1.0):")
share_cols = [c for c in category_features.columns if c.endswith('_share')]
print(category_features[share_cols].sum(axis=1).describe())

                             0
CustomerID           BSB060048
Individual                   Y
grocery_share         0.752791
dining_share          0.029003
gas_share             0.016285
digital_share              0.0
telecom_share              0.0
retail_share          0.192412
healthcare_share       0.00435
pets_share                 0.0
home_share            0.005159
automotive_share           0.0
clothing_share             0.0
entertainment_share        0.0
financial_share            0.0
personal_care_share        0.0
transport_share            0.0
utilities_share            0.0
government_share           0.0
other_share                0.0
Shape: (68585, 20)

Share totals per customer (should all be ~1.0):
count    68585.000000
mean         0.992789
std          0.051152
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
dtype: float64


99.27% customers account to 1 after adding all their shares \
min = 0 (108 customers having no shares) 

Temporal features

In [5]:
# To calculate when customers spend
temporal_features = con.execute("""
    SELECT 
        t.CustomerID,
        c.Individual,
        -- Weekend share
        SUM(CASE WHEN DAYOFWEEK(t.Transaction_Date) IN (0, 6) 
            THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS weekend_share,
        
        -- Avg transactions per week (~10 week window)
        COUNT(*) / 10.0 AS avg_transactions_per_week,
        
        -- Daily spend ratio using active days for Nov-Dec
        (SUM(CASE WHEN MONTH(t.Transaction_Date) = 12 
            THEN t.Amount_Completed ELSE 0 END) 
        / NULLIF(COUNT(DISTINCT CASE WHEN MONTH(t.Transaction_Date) = 12 
            THEN t.Transaction_Date END), 0))
        /
        NULLIF(
            SUM(CASE WHEN MONTH(t.Transaction_Date) = 11 
                THEN t.Amount_Completed ELSE 0 END) 
            / NULLIF(COUNT(DISTINCT CASE WHEN MONTH(t.Transaction_Date) = 11 
                THEN t.Transaction_Date END), 0)
        , 0) AS dec_to_nov_daily_spend_ratio,
                                
        -- Daily spend ratio using active days for Dec-Jan
        (SUM(CASE WHEN MONTH(t.Transaction_Date) = 1 
            THEN t.Amount_Completed ELSE 0 END) 
        / NULLIF(COUNT(DISTINCT CASE WHEN MONTH(t.Transaction_Date) = 1 
            THEN t.Transaction_Date END), 0))
        /
        NULLIF(
            SUM(CASE WHEN MONTH(t.Transaction_Date) = 12 
                THEN t.Amount_Completed ELSE 0 END) 
            / NULLIF(COUNT(DISTINCT CASE WHEN MONTH(t.Transaction_Date) = 12 
                THEN t.Transaction_Date END), 0)
        , 0) AS jan_to_dec_daily_spend_ratio

    FROM transactions_v2 t
    INNER JOIN customers c ON t.CustomerID = c.CustomerID
    WHERE t.Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25'
      AND t.Amount_Completed > 0
    GROUP BY t.CustomerID, c.Individual
""").fetchdf()

print("Shape:", temporal_features.shape)
print("\n", temporal_features.describe())


Shape: (68585, 6)

        weekend_share  avg_transactions_per_week  dec_to_nov_daily_spend_ratio  \
count   68585.000000               68585.000000                  55480.000000   
mean        0.272045                   6.288897                      2.445252   
std         0.168781                   6.368118                     32.958771   
min         0.000000                   0.100000                      0.001308   
25%         0.186047                   1.200000                      0.788309   
50%         0.266667                   4.400000                      1.171436   
75%         0.342105                   9.500000                      1.847944   
max         1.000000                 142.400000                   6217.375000   

       jan_to_dec_daily_spend_ratio  
count                  60316.000000  
mean                       1.333959  
std                       20.577821  
min                        0.000175  
25%                        0.605678  
50%                   

On average, 27% transactions happen on weekends \
Mean drops in Jan compared to Dec and so does median


Merchant categories

In [6]:
# to calculate merchant categories 
merchant_features = con.execute("""
    SELECT 
        CustomerID,
        Individual,
        COUNT(DISTINCT Merchant_Name) AS unique_merchants,
        
        MAX(merchant_total) / SUM(merchant_total) AS top_merchant_share,
        
        -- what fraction of merchants got a repeat visit
        SUM(CASE WHEN merchant_visits > 1 THEN 1 ELSE 0 END) * 1.0 
            / COUNT(*) AS repeat_merchant_rate,
        
        -- what fraction of transactions are repeat visits
        SUM(CASE WHEN merchant_visits > 1 THEN merchant_visits - 1 ELSE 0 END) * 1.0 
            / SUM(merchant_visits) AS repeat_transaction_rate,
        
        SUM(merchant_visits) * 1.0 / COUNT(*) AS avg_visits_per_merchant
        
    FROM (
        SELECT 
            t.CustomerID,
            c.Individual,
            t.Merchant_Name,
            COUNT(*) AS merchant_visits,
            SUM(t.Amount_Completed) AS merchant_total
        FROM transactions_v2 t
        INNER JOIN customers c on t.CustomerID = c.CustomerID
        WHERE t.Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25'
          AND t.Amount_Completed > 0
        GROUP BY t.CustomerID, c.Individual, t.Merchant_Name
    ) sub
    GROUP BY CustomerID, Individual
""").fetchdf()

print(merchant_features.shape)
print(merchant_features.describe())

(68585, 7)
       unique_merchants  top_merchant_share  repeat_merchant_rate  \
count       68585.00000        68585.000000          68585.000000   
mean           29.67732            0.371103              0.352530   
std            27.52562            0.267817              0.219693   
min             1.00000            0.017134              0.000000   
25%             7.00000            0.171228              0.230769   
50%            23.00000            0.275472              0.333333   
75%            45.00000            0.491557              0.450980   
max           522.00000            1.000000              1.000000   

       repeat_transaction_rate  avg_visits_per_merchant  
count             68585.000000             68585.000000  
mean                  0.421053                 2.099583  
std                   0.217518                 1.542676  
min                   0.000000                 1.000000  
25%                   0.291667                 1.411765  
50%                

On average, 30 unique merchants \
37% top_merchant_share

Demographic features

In [7]:
demo_features = con.execute("""
    SELECT 
        c.CustomerID,
        c.Individual,
        c.Age,
        c.RelationshipYears,
        
        -- Gender encoding
        CASE WHEN c.Gender = 'M' THEN 1 ELSE 0 END AS is_male,
        CASE WHEN c.Gender = 'F' THEN 1 ELSE 0 END AS is_female,
        
        -- Account counts
        c.NumberActiveDDAs,
        c.NumberActiveTimeDeposits,
        c.NumberActiveLoans,
        c.NumberCreditCardAccts,
        
        -- Binary flags (Y/N → 1/0)
        CASE WHEN c.DepositAccount = 'Y' THEN 1 ELSE 0 END AS has_deposit,
        CASE WHEN c.LoanAccount = 'Y' THEN 1 ELSE 0 END AS has_loan,
        CASE WHEN c.CreditCardAccount = 'Y' THEN 1 ELSE 0 END AS has_creditcard,
        CASE WHEN c.ActiveATMCard = 'Y' THEN 1 ELSE 0 END AS has_atm,
        CASE WHEN c.ActiveDebitCard = 'Y' THEN 1 ELSE 0 END AS has_debit,
        CASE WHEN c.PrimaryBankingCustomerFlag = 'Y' THEN 1 ELSE 0 END AS is_primary,
        CASE WHEN c.BangorWealth = 'Yes' THEN 1 ELSE 0 END AS is_wealth,
        CASE WHEN c.Payroll = 'Yes' THEN 1 ELSE 0 END AS is_payroll,
        CASE WHEN c.PlingCustomer = 'Yes' THEN 1 ELSE 0 END AS is_pling,
        CASE WHEN c.ABLECustomer = 'Yes' THEN 1 ELSE 0 END AS is_able
        
    FROM customers c
    INNER JOIN (SELECT DISTINCT CustomerID
                FROM transactions_v2
                WHERE Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25') t ON c.CustomerID = t.CustomerID
""").fetchdf()

print("Shape:", demo_features.shape)
print("\n", demo_features.describe())


Shape: (68857, 20)

                 Age  RelationshipYears       is_male     is_female  \
count  68857.000000       68847.000000  68857.000000  68857.000000   
mean      42.169511          13.327407      0.376302      0.364349   
std       25.432327          11.470189      0.484461      0.481251   
min        0.000000           0.000000      0.000000      0.000000   
25%       24.000000           4.000000      0.000000      0.000000   
50%       43.000000          10.000000      0.000000      0.000000   
75%       63.000000          20.000000      1.000000      1.000000   
max      126.000000          85.000000      1.000000      1.000000   

       NumberActiveDDAs  NumberActiveTimeDeposits  NumberActiveLoans  \
count      68857.000000              68857.000000       68857.000000   
mean           1.857647                  0.085423           0.265333   
std            1.436369                  0.489899           0.597602   
min            0.000000                  0.000000           

In [8]:
trend_features = con.execute("""
    SELECT 
        t.CustomerID,
        c.Individual,
                             
        -- Average transaction size change (Dec avg / Nov avg)
        (SUM(CASE WHEN MONTH(t.Transaction_Date) = 12 THEN t.Amount_Completed ELSE 0 END) 
         / NULLIF(COUNT(CASE WHEN MONTH(t.Transaction_Date) = 12 THEN 1 END), 0))
        /
        NULLIF(
            SUM(CASE WHEN MONTH(t.Transaction_Date) = 11 THEN t.Amount_Completed ELSE 0 END) 
            / NULLIF(COUNT(CASE WHEN MONTH(t.Transaction_Date) = 11 THEN 1 END), 0)
        , 0) AS dec_to_nov_avg_amount_ratio,
                             
        -- Average transaction size change (Jan avg / Dec avg)
        (SUM(CASE WHEN MONTH(t.Transaction_Date) = 1 THEN t.Amount_Completed ELSE 0 END) 
        / NULLIF(COUNT(CASE WHEN MONTH(t.Transaction_Date) = 1 THEN 1 END), 0))
        /
        NULLIF(
            SUM(CASE WHEN MONTH(t.Transaction_Date) = 12 THEN t.Amount_Completed ELSE 0 END) 
            / NULLIF(COUNT(CASE WHEN MONTH(t.Transaction_Date) = 12 THEN 1 END), 0)
        , 0) AS jan_to_dec_avg_amount_ratio

    FROM transactions_v2 t
    INNER JOIN customers c ON t.CustomerID = c.CustomerID
    WHERE t.Transaction_Date BETWEEN '2025-11-01' AND '2026-01-25'
      AND t.Amount_Completed > 0
    GROUP BY t.CustomerID, c.Individual
""").fetchdf()

print("Shape:", trend_features.shape)
print("\n", trend_features.describe())

Shape: (68585, 4)

        dec_to_nov_avg_amount_ratio  jan_to_dec_avg_amount_ratio
count                 55480.000000                 60316.000000
mean                      1.880435                     1.371670
std                      16.055016                    20.435305
min                       0.001308                     0.000611
25%                       0.784725                     0.706926
50%                       1.090231                     0.958024
75%                       1.587107                     1.252011
max                    2925.823529                  4861.000000


In [9]:
# Drop Individual from all except spending_agg
category_features = category_features.drop(columns=['Individual'])
temporal_features = temporal_features.drop(columns=['Individual'])
merchant_features = merchant_features.drop(columns=['Individual'])
demo_features = demo_features.drop(columns=['Individual'])
trend_features = trend_features.drop(columns=['Individual'])


# Now merge
features = spending_agg.copy()
features = features.merge(category_features, on='CustomerID', how='inner')
features = features.merge(temporal_features, on='CustomerID', how='inner')
features = features.merge(merchant_features, on='CustomerID', how='inner')
features = features.merge(demo_features, on='CustomerID', how='inner')
features = features.merge(trend_features, on='CustomerID', how='inner')

print("Final shape:", features.shape)
print("\nNull counts:")
print(features.isnull().sum()[features.isnull().sum() > 0])

Final shape: (68585, 56)

Null counts:
std_transaction_amount           3032
dec_to_nov_daily_spend_ratio    13105
jan_to_dec_daily_spend_ratio     8269
RelationshipYears                  10
dec_to_nov_avg_amount_ratio     13105
jan_to_dec_avg_amount_ratio      8269
dtype: int64


3032 customers made only 1 transaction (impute with 0) \
13105 customers made transaction either in nov or dec (impute with 1) \
8269 customers made transaction either in dec or jan (impute with 1) \
10 customers having no relationship years (impute with median)



In [10]:
# Impute NULLs
features['std_transaction_amount'] = features['std_transaction_amount'].fillna(0)
features['RelationshipYears'] = features['RelationshipYears'].fillna(features['RelationshipYears'].median())

# Ratio features - 1.0 means no change
ratio_cols = ['dec_to_nov_daily_spend_ratio', 'jan_to_dec_daily_spend_ratio',
              'dec_to_nov_avg_amount_ratio', 'jan_to_dec_avg_amount_ratio']
features[ratio_cols] = features[ratio_cols].fillna(1.0)

# Verify
print(features.isnull().sum()[features.isnull().sum() > 0])

Series([], dtype: int64)


In [11]:
print("Final shape:", features.shape)
print("\nNull counts:")
print(features.isnull().sum()[features.isnull().sum() > 0])

Final shape: (68585, 56)

Null counts:
Series([], dtype: int64)


In [16]:
# Save to CSV
features.to_csv('../data/customer_features.csv', index=False)

# Load into DuckDB
con.execute("DROP TABLE IF EXISTS customer_features")
con.execute("""
    CREATE TABLE customer_features AS 
    SELECT * FROM read_csv_auto('../data/customer_features.csv')
""")

print(con.execute("SELECT COUNT(*) FROM customer_features").fetchdf())

   count_star()
0         68585


In [17]:
# Check table
print(con.execute("SELECT COUNT(*) FROM customer_features").fetchdf())

# Check columns
print(con.execute("DESCRIBE customer_features").fetchdf())

   count_star()
0         68585
                     column_name column_type null   key default extra
0                     CustomerID     VARCHAR  YES  None    None  None
1                     Individual     VARCHAR  YES  None    None  None
2                    total_spend      DOUBLE  YES  None    None  None
3              transaction_count      BIGINT  YES  None    None  None
4         avg_transaction_amount      DOUBLE  YES  None    None  None
5      median_transaction_amount      DOUBLE  YES  None    None  None
6         std_transaction_amount      DOUBLE  YES  None    None  None
7         max_transaction_amount      DOUBLE  YES  None    None  None
8         min_transaction_amount      DOUBLE  YES  None    None  None
9                  grocery_share      DOUBLE  YES  None    None  None
10                  dining_share      DOUBLE  YES  None    None  None
11                     gas_share      DOUBLE  YES  None    None  None
12                 digital_share      DOUBLE  YES  None   

In [15]:
print(features.head(1).T)

                                      0
CustomerID                    BSB382953
Individual                            Y
total_spend                     1915.99
transaction_count                    70
avg_transaction_amount        27.371286
median_transaction_amount        14.815
std_transaction_amount        50.850889
max_transaction_amount           389.97
min_transaction_amount             1.04
grocery_share                  0.205925
dining_share                   0.101389
gas_share                      0.097073
digital_share                  0.084927
telecom_share                       0.0
retail_share                   0.032735
healthcare_share                    0.0
pets_share                          0.0
home_share                          0.0
automotive_share                    0.0
clothing_share                      0.0
entertainment_share            0.389746
financial_share                     0.0
personal_care_share            0.017745
transport_share                     0.0


In [18]:
con.close()